In [1]:
using Revise

In [2]:
using GeneRegulatorySystems
using GeneRegulatorySystems.Models
using GeneRegulatorySystems.Models.NetworkRepresentation
using GeneRegulatorySystems.Models.V1
using GeneRegulatorySystems.Scheduling
import JSON
using GarishPrint

macro pp(x) :(GarishPrint.pprint($(esc(x)))) end;

Load schedule

In [82]:
path_diff = "examples/specification/differentiation.schedule.json"
path_v1 = "examples/toy/ACDC.schedule.json"
path_kronecker = "examples/benchmark/kronecker-small.schedule.json"
path_rand_diff = "examples/specification/random-differentiation.schedule.json"

schedule! = Models.load(path_v1, seed="seed123");

In [ ]:
f! = Scheduling.reify(schedule!, "-2-1+-1+")

In [ ]:
rs = f!.f!.model.model.definition

In [ ]:
using Catalyst
using GraphMakie
using NetworkLayout
using CairoMakie

Catalyst.plot_network(rs)

Reify the schedule. Need to do a dryrun to get all the primitives

In [ ]:
# this only extracts the first network that it finds in the schedule, rather than all of them
function extract_network(schedule::Scheduling.Schedule)
    network = nothing
    
    function dryrun_collector(primitive!, x, Δt; path, _...)
        if !(isfinite(Δt) && Δt > 0.0)
            return
        end
        if network === nothing
            network = Models.NetworkRepresentation.entity(primitive!)
        end
    end
    
    schedule(Models.FlatState(); dryrun=dryrun_collector)
    return network
end

network = extract_network(schedule!)

In [ ]:
schedule_files = [
    "examples/specification/differentiation.schedule.json",
    "examples/benchmark/differentiated-small.schedule.json",
    "examples/benchmark/differentiated-medium.schedule.json",
    "examples/benchmark/kronecker-tiny.schedule.json",
    "examples/benchmark/kronecker-small.schedule.json",
    "examples/benchmark/kronecker-medium.schedule.json",
    "examples/benchmark/kronecker-big.schedule.json",
    #"examples/benchmark/kronecker-huge.schedule.json",
]

results = Dict()
timings = Dict()

for path in schedule_files
    try
        @info "Processing: $path"
        elapsed = @elapsed begin
            sched = Models.load(path, seed="seed123")
            net = extract_network(sched)
        end
        
        if net !== nothing
            flat_nodes, flat_links = NetworkRepresentation.flatten(net)
            num_nodes = length(flat_nodes)
            num_links = length(flat_links)
            @info "  Success: $(num_nodes) nodes, $(num_links) links ($(round(elapsed, digits=3))s)"
            results[path] = "Success (nodes=$num_nodes, links=$num_links)"
            timings[path] = (time=elapsed, nodes=num_nodes)
        else
            @warn "  Failed: network is nothing"
            results[path] = "Failed"
            timings[path] = (time=elapsed, nodes=0)
        end
    catch e
        @error "  Error: $e"
        results[path] = "Error: $(e)"
        timings[path] = (time=NaN, nodes=0)
    end
end

In [ ]:
using CairoMakie

nodes_list = [timings[path].nodes for path in schedule_files if !isnan(timings[path].time)]
times_list = [timings[path].time for path in schedule_files if !isnan(timings[path].time)]
labels_list = [splitdir(path)[2] for path in schedule_files if !isnan(timings[path].time)]

fig = Figure(size=(1000, 600))
ax = Axis(fig[1, 1], xlabel="Number of Nodes", ylabel="Time (s)", 
          title="Processing Time vs Network Size")

scatter!(ax, nodes_list, times_list, markersize=10, color=:blue, alpha=0.6)

for (x, y, label) in zip(nodes_list, times_list, labels_list)
    text!(ax, x, y + 0.01, text=label, fontsize=9)
end

fig

In [ ]:
using GeneRegulatorySystems.Models: Wrapped, Instant, Label, Descriptions

_label(wrapped::Wrapped) = _label(Models.describe(wrapped.definition))

_label(label::Label) = label.label

_label(model::Instant) = repr(model)

function _label(desc::Descriptions)
    i = findfirst(d -> d isa Label, desc.descriptions)
    if i !== nothing
        _label(desc.descriptions[i])
    else
        ""
    end
end

function get_segments(schedule::Scheduling.Schedule)
    segments = []
    function dryrun_collector(primitive!, x, Δt; path, _...)
        label = _label(primitive!.f!.model)
        push!(segments, (path = path, from = x.t, to = x.t + Δt, label = label))
    end
    
    schedule(Models.FlatState(); dryrun=dryrun_collector)
    return segments
end

segments = get_segments(schedule!)

GeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' networ

Excessive output truncated after 524325 bytes.

Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySystems.Models.V1.Definition'regulation/v1' network with 3 genesGeneRegulatorySyst

17001-element Vector{Any}:
 (path = "-1", from = 0.0, to = 0.0, label = "GeneRegulatorySystems.Models.Plumbing.Adjust(+, Dict(:polymerases => 500000, Symbol(\"3.proteins\") => 20, :proteasomes => 1000000, Symbol(\"1.proteins\") => 20, Symbol(\"2.proteins\") => 20, :ribosomes => 2000000))")
 (path = "-2-1/1+", from = 0.0, to = 1000.0, label = "'regulation/v1' network with 3 genes")
 (path = "-2-1/1+", from = 1000.0, to = 2000.0, label = "'regulation/v1' network with 3 genes")
 (path = "-2-1/1+", from = 2000.0, to = 3000.0, label = "'regulation/v1' network with 3 genes")
 (path = "-2-1/1+", from = 3000.0, to = 4000.0, label = "'regulation/v1' network with 3 genes")
 (path = "-2-1/1+", from = 4000.0, to = 5000.0, label = "'regulation/v1' network with 3 genes")
 (path = "-2-1/1+", from = 5000.0, to = 6000.0, label = "'regulation/v1' network with 3 genes")
 (path = "-2-1/1+", from = 6000.0, to = 7000.0, label = "'regulation/v1' network with 3 genes")
 (path = "-2-1/1+", from = 7000.0, to = 